# Lab 2: Packet Analysis and Network Traffic Inspection
**CMSC395 — Computer Networks**  
Estimated time: 3–4 hours | Points: 100

---

Work through each cell in order. Run a cell with **Shift+Enter** and read the output before moving on.

> **Submission:** Use **Kernel → Restart Kernel and Run All Cells** before your final push.  
> Commit `lab2_my_capture.pcap` to your repository — it is a required artifact.

## Setup — Imports

Run this cell first.

In [ ]:
# Setup — run this cell before starting any part
import subprocess
import csv
import json
import time
import socket
import os
from collections import defaultdict
from pathlib import Path

import pyshark
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Paths - do not change these
PROVIDED_PCAP = Path('lab2_provided.pcap')
MY_PCAP       = Path('lab2_my_capture.pcap')
MY_FIELDS_CSV = Path('lab2_fields.csv')

LAB_SERVER = 'lab.millersville.edu'
LAB_HTTP_PORT = 8080

# Sanity check
if not PROVIDED_PCAP.exists():
    print(f'WARNING: {PROVIDED_PCAP} not found. Re-run the nbgitpuller link to fetch it')
else:
    print(f'Provided capture: {PROVIDED_PCAP} ({PROVIDED_PCAP.stat().st_size:,} bytes)')

print('Imports OK')

---
## Part 1: Wireshark and tshark Orientation

Answer the five questions in the lab guide using both `tshark` (programmatically in Cell 1.1)
and Wireshark (described in Cell 1.2).  
All tshark commands must be run via `subprocess`. Do not paste terminal screenshots.

### Cell 1.1 — tshark Queries

Use `subprocess.run` to execute each `tshark` command and capture its output.  
Print the command and its output for each of the five questions.

Example pattern:
```python
result = subprocess.run(
    ['tshark', '-r', str(PROVIDED_PCAP), '-Y', 'tcp', '-T', 'fields', '-e', 'tcp.stream', '-E', 'header=y'],
    capture_output=True, text=True
)
print(result.stdout)
```

In [ ]:
# Cell 1.1 - tshark queries for all 5 questions
# Label each section clearly: Q1, Q2, Q3, Q4, Q5

def tshark(args, pcap=PROVIDED_PCAP):
    """Run tshark with the given args list against the provided pcap. Returns stdout as string."""
    result = subprocess.run(
        ['tshark', '-r', str(pcap)] + args,
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'tshark error: {result.stderr[:200]}')
    return result.stdout


# Q1: Unique IP addresses
print('=== Q1: Unique IP addresses ===')
# Your tshark command here


# Q2: TCP streams
print('\n=== Q2: TCP streams ===')
# Your tshark command here


# Q3: First TCP handshake RTT
print('\n=== Q3: First TCP handshake RTT ===')
# Your tshark command here


# Q4: DNS queries and TTLs
print('\n=== Q4: DNS queries and TTLs ===')
# Your tshark command here


# Q5: TCP retransmissions
print('\n=== Q5: TCP retransmissions ===')
# Your tshark command here


### Cell 1.2 — Wireshark Approach

For each question, describe how you located the answer in Wireshark.  
Include the display filter you applied and which column or panel you used to read the answer.  
Double-click this cell to edit it.

**Q1. Unique IP addresses:**  
*(Describe your Wireshark approach here)*

**Q2. TCP streams:**  
*(Describe your Wireshark approach here)*

**Q3. First TCP handshake RTT:**  
*(Describe your Wireshark approach here)*

**Q4. DNS queries and TTLs:**  
*(Describe your Wireshark approach here)*

**Q5. TCP retransmissions:**  
*(Describe your Wireshark approach here)*

### Reflection 1.A

For which of the five questions was the `tshark` command-line approach faster or more reliable than the GUI?
For which was Wireshark more useful? What does this tell you about when to use each tool?

**Your answer:**

*(Write here)*

### Reflection 1.B

You found retransmissions in the capture. From the timestamps, how long after the original
transmission did the retransmission occur? What determines this timeout value, and who controls it?

**Your answer:**

*(Write here)*

---
## Part 2: Programmatic Analysis with pyshark

Write a Python analysis script that reads `lab2_provided.pcap` and produces a structured report.
Each value must be computed programmatically — do not hardcode anything you read from Wireshark.

### Cell 2.1 — Packet Size Histogram

Extract the IP payload size for every packet. Plot a histogram with 50-byte bins.  
Report mean, median, and 99th percentile. In the markdown cell below the plot,
describe any bimodal structure you observe and explain what generates each peak.

**Hint:** The IP payload size is `pkt.ip.len` minus the IP header length (`pkt.ip.hdr_len`).

In [ ]:
# Cell 2.1 — Packet size histogram

sizes = []

cap = pyshark.FileCapture(str(PROVIDED_PCAP), display_filter='ip')
for pkt in cap:
    try:
        # Your code here — extract packet size and append to sizes
        pass
    except AttributeError:
        pass
cap.close()

print(f'Packets analyzed: {len(sizes)}')
print(f'Mean size:        {np.mean(sizes):.1f} bytes')
print(f'Median size:      {np.median(sizes):.1f} bytes')
print(f'99th percentile:  {np.percentile(sizes, 99):.1f} bytes')

# Plot histogram
fig, ax = plt.subplots(figsize=(10, 4))
# Your plotting code here
plt.tight_layout()
plt.show()


**Histogram description:**

*(Describe the distribution. What causes any peaks you observe?)*

### Cell 2.2 — Top Flows by Volume

Identify the top 5 flows by total bytes transferred. A flow is identified by its 5-tuple:
(src IP, src port, dst IP, dst port, protocol). Display as a formatted table showing
rank, 5-tuple, total bytes, and packet count.

**Hint:** `pkt.ip.proto` gives the IP protocol number (6=TCP, 17=UDP).  
For TCP: `pkt.tcp.srcport`, `pkt.tcp.dstport`.  
For UDP: `pkt.udp.srcport`, `pkt.udp.dstport`.

In [ ]:
# Cell 2.2 - Top flows by volume

flow_bytes  = defaultdict(int)
flow_pkts   = defaultdict(int)

cap = pyshark.FileCapture(str(PROVIDED_PCAP), display_filter='ip')
for pkt in cap:
    try:
        # Build the 5-tuple and accumulate bytes/packets
        # Your code here
        pass
    except AttributeError:
        pass
cap.close()

# Sort and display top 5
top5 = sorted(flow_bytes.items(), key=lambda x: x[1], reverse=True)[:5]

print(f"{'Rank':<5} {'Flow (src:port -> dst:port proto)':<50} {'Bytes':>10} {'Packets':>8}")
print('-' * 78)
for rank, (flow, total_bytes) in enumerate(top5, 1):
    src_ip, src_port, dst_ip, dst_port, proto = flow
    flow_str = f"{src_ip}:{src_port} -> {dst_ip}:{dst_port} ({proto})"
    print(f"{rank:<5} {flow_str:<50} {total_bytes:>10,} {flow_pkts[flow]:>8,}")


### Cell 2.3 — DNS Response Times

Match each DNS query with its response using the DNS transaction ID (`dns.id`).
Compute the time between query and response for each transaction.
Display a summary table: hostname queried, query time, response time, response delay (ms), and TTL.
Note any transactions that took significantly longer than the others.

In [ ]:
# Cell 2.3 — DNS response times

dns_queries   = {}   # transaction_id -> {name, time}
dns_responses = {}   # transaction_id -> {time, ttl, answer}

cap = pyshark.FileCapture(str(PROVIDED_PCAP), display_filter='dns')
for pkt in cap:
    try:
        txid = pkt.dns.id
        t    = float(pkt.frame_info.time_relative)
        is_response = int(pkt.dns.flags_response)

        if not is_response:
            # This is a query - record the name and time
            # Your code here
            pass
        else:
            # This is a response - record time, TTL, and answer
            # Your code here
            pass
    except AttributeError:
        pass
cap.close()

# Match queries to responses and compute delays
print(f"{'Hostname':<35} {'Query (s)':>10} {'Resp (s)':>10} {'Delay (ms)':>12} {'TTL':>6}")
print('-' * 80)

delays = []
for txid, q in dns_queries.items():
    if txid in dns_responses:
        r = dns_responses[txid]
        delay_ms = (r['time'] - q['time']) * 1000
        delays.append(delay_ms)
        name = q.get('name', '?')[:34]
        print(f"{name:<35} {q['time']:>10.4f} {r['time']:>10.4f} {delay_ms:>12.2f} {r.get('ttl','?'):>6}")

if delays:
    print(f"\nMean response time: {np.mean(delays):.2f} ms")
    print(f"Max response time:  {np.max(delays):.2f} ms")


### Cell 2.4 — Retransmission Analysis

Find all TCP retransmissions in the capture. For each show:
stream index, sequence number, time of original transmission,
time of retransmission, and retransmission delay (ms).

Compare the retransmission delay to the RTT of that stream. Is the timeout proportional to RTT?

**Hint:** `tshark` marks retransmitted packets with the field `tcp.analysis.retransmission`.  
Use `display_filter='tcp.analysis.retransmission'` in pyshark to find them directly.

In [ ]:
# Cell 2.4 - Retransmission analysis

retransmissions = []

# Pass 1: find all retransmitted packets
cap = pyshark.FileCapture(str(PROVIDED_PCAP),
                           display_filter='tcp.analysis.retransmission')
for pkt in cap:
    try:
        retransmissions.append({
            'stream':  int(pkt.tcp.stream),
            'seq':     int(pkt.tcp.seq),
            'time':    float(pkt.frame_info.time_relative),
            'frame':   int(pkt.frame_info.number),
        })
    except AttributeError:
        pass
cap.close()

print(f'Total retransmissions found: {len(retransmissions)}')

# Pass 2: find the original transmission time for each retransmitted seq number
# Your code here - match each retransmission to its original packet
# Hint: iterate over all TCP packets and record the first time each (stream, seq) pair is seen


# Display results table
print(f"\n{'Stream':>7} {'Seq':>12} {'Original (s)':>14} {'Retrans (s)':>12} {'Delay (ms)':>12}")
print('-' * 65)
# Your display code here


### Cell 2.5 — Stream Timeline

For each TCP stream compute: start time, end time, duration (ms), and total bytes transferred.
Display as a table sorted by start time. Identify which streams overlapped (parallel connections)
and annotate each stream with what application content it likely carried
(infer from port numbers and payload size).

In [ ]:
# Cell 2.5 - Stream timeline

streams = defaultdict(lambda: {
    'start': float('inf'),
    'end':   0.0,
    'bytes': 0,
    'pkts':  0,
    'src_port': None,
    'dst_port': None,
})

cap = pyshark.FileCapture(str(PROVIDED_PCAP), display_filter='tcp')
for pkt in cap:
    try:
        sid = int(pkt.tcp.stream)
        t   = float(pkt.frame_info.time_relative)
        # Your code here - update stream stats
        pass
    except AttributeError:
        pass
cap.close()

# Display sorted by start time
sorted_streams = sorted(streams.items(), key=lambda x: x[1]['start'])

print(f"{'Stream':>7} {'Start (s)':>10} {'End (s)':>10} {'Duration (ms)':>14} "
      f"{'Bytes':>10} {'Src Port':>9} {'Dst Port':>9}")
print('-' * 80)

for sid, s in sorted_streams:
    duration_ms = (s['end'] - s['start']) * 1000
    print(f"{sid:>7} {s['start']:>10.4f} {s['end']:>10.4f} {duration_ms:>14.1f} "
          f"{s['bytes']:>10,} {str(s['src_port']):>9} {str(s['dst_port']):>9}")

# Identify overlapping streams
print('\nOverlapping stream pairs:')
# Your code here


### Reflection 2.A

Your packet size histogram likely shows a bimodal distribution. What are the two populations
of packets and what generates each? What would the distribution look like on a link carrying
only bulk file transfer traffic?

**Your answer:**

*(Write here)*

### Reflection 2.B

Look at the retransmission delays relative to the RTTs of those streams. What pattern do you observe?
What algorithm determines when a TCP sender decides a packet has been lost and retransmits it?

**Your answer:**

*(Write here)*

### Reflection 2.C

Your stream timeline shows which connections were parallel. Why does a browser open multiple
TCP connections to the same server rather than sending all requests on one connection?
What limitation does this work around?

**Your answer:**

*(Write here)*

---
## Part 3: Capture Your Own Traffic

Generate your own packet capture using `tshark` while running the session script below
against the lab server. Analyze the result.

**Before running Cell 3.1**, open a terminal and start the capture:

```bash
tshark -i any -w ~/lab2_my_capture.pcap -f "host lab.millersville.edu" -a duration:60
```

Then run Cell 3.1. After it completes, copy the capture into your repo directory:

```bash
cp ~/lab2_my_capture.pcap ./lab2_my_capture.pcap
```

Commit the file:

```bash
git add lab2_my_capture.pcap
git commit -m "Lab2: add personal capture file"
```

### Cell 3.1 — Session Script (pre-written)

This cell is pre-written. Run it while `tshark` is capturing in the terminal.  
The printed timestamps are part of your submission. Leave them visible.

In [ ]:
# Cell 3.1 — Session script (pre-written - do not modify)
# Run tshark in a terminal BEFORE running this cell.

import urllib.request   # Used here only - your analysis cells must use raw sockets

def ts():
    """Return current time as a formatted string."""
    return f"{time.perf_counter():.3f}s"

base = f"http://{LAB_SERVER}:{LAB_HTTP_PORT}"

print(f"[{ts()}] Session start")

# Step 1: DNS resolution (explicit)
print(f"[{ts()}] Resolving {LAB_SERVER}...")
resolved_ip = socket.gethostbyname(LAB_SERVER)
print(f"[{ts()}] Resolved to {resolved_ip}")

# Step 2: HTTP request to / (expect redirect)
print(f"[{ts()}] GET / (expect redirect)")
try:
    resp = urllib.request.urlopen(f"{base}/", timeout=5)
    print(f"[{ts()}] Final URL after redirect: {resp.url}  status={resp.status}")
except Exception as e:
    print(f"[{ts()}] Request failed: {e}")

time.sleep(2)

# Step 3: Fetch TCP RFC
print(f"[{ts()}] GET /rfc/rfc793.txt")
try:
    resp = urllib.request.urlopen(f"{base}/rfc/rfc793.txt", timeout=10)
    data = resp.read()
    print(f"[{ts()}] Received {len(data):,} bytes")
except Exception as e:
    print(f"[{ts()}] Request failed: {e}")

time.sleep(2)

# Step 4: Repeat fetch (test DNS caching)
print(f"[{ts()}] GET /rfc/rfc793.txt (repeat — test DNS caching)")
try:
    resp = urllib.request.urlopen(f"{base}/rfc/rfc793.txt", timeout=10)
    data = resp.read()
    print(f"[{ts()}] Received {len(data):,} bytes")
except Exception as e:
    print(f"[{ts()}] Request failed: {e}")

time.sleep(1)

# Step 5: Slow endpoint
print(f"[{ts()}] GET /slow (chunked response)")
try:
    resp = urllib.request.urlopen(f"{base}/slow", timeout=30)
    data = resp.read()
    print(f"[{ts()}] Received {len(data):,} bytes")
except Exception as e:
    print(f"[{ts()}] Request failed: {e}")

print(f"[{ts()}] Session complete")


### Export Fields to CSV

After copying `lab2_my_capture.pcap` to your repository directory, run this cell
to export key fields for analysis.

In [ ]:
# Export fields from personal capture to CSV. Run after copying pcap

if not MY_PCAP.exists():
    print(f'ERROR: {MY_PCAP} not found.')
    print('Copy it from your home directory first:')
    print('  cp ~/lab2_my_capture.pcap ./lab2_my_capture.pcap')
else:
    result = subprocess.run([
        'tshark', '-r', str(MY_PCAP),
        '-T', 'fields',
        '-e', 'frame.number',
        '-e', 'frame.time_relative',
        '-e', 'ip.src',
        '-e', 'ip.dst',
        '-e', 'ip.proto',
        '-e', 'tcp.stream',
        '-e', 'tcp.len',
        '-e', 'tcp.seq',
        '-e', 'tcp.flags',
        '-e', 'dns.qry.name',
        '-e', 'dns.resp.ttl',
        '-E', 'header=y',
        '-E', 'separator=,',
    ], capture_output=True, text=True)

    MY_FIELDS_CSV.write_text(result.stdout)
    lines = result.stdout.count('\n')
    print(f'Exported {lines - 1} packets to {MY_FIELDS_CSV}')


### Cell 3.2 - Capture Analysis

Load your personal capture and answer the four questions below.
Each answer must reference specific packet numbers or timestamps from **your** capture.

In [ ]:
# Cell 3.2 — Capture analysis
# Answer each question with code + a markdown cell below

if not MY_PCAP.exists():
    print('lab2_my_capture.pcap not found. Complete the capture steps first')
else:

    # Q1: How many TCP connections were opened?
    print('=== Q1: TCP connections ===')
    # Your code here


    # Q2: Did the repeat RFC fetch trigger a new DNS query?
    print('\n=== Q2: DNS caching check ===')
    # Your code here


    # Q3: Sequence number vs time plot for /slow stream
    print('\n=== Q3: /slow stream — sequence number plot ===')
    # Identify the /slow stream (hint: it has the longest duration)
    # Plot seq number vs time for that stream
    # Your code here


    # Q4: Time between first response and second request (redirect)
    print('\n=== Q4: Redirect timing ===')
    # Your code here


**Q1. TCP connections:** *(Write your answer with specific packet references)*

**Q2. DNS caching:** *(Did you see a new DNS query for the repeat fetch? What TTL did you observe?)*

**Q3. /slow stream:** *(Describe the shape of the sequence number curve)*

**Q4. Redirect timing:** *(What was the gap between the first response and second request?)*

### Cell 3.3 — Timestamp Correlation

Your session script printed timestamps when each request started and completed.
Correlate those with your capture:

1. For each request, compute the gap between your script's send timestamp and the first packet in the capture. What does this gap represent?
2. Compare your script's application-layer RTT with the TCP-level RTT visible in the capture. Are they the same? If not, why not?

Hardcode the timestamps from your Cell 3.1 output into the cell below.

In [ ]:
# Cell 3.3 — Timestamp correlation

# Paste your timestamps from Cell 3.1 output here
# Example — replace with your actual values:
script_timestamps = {
    'session_start':   0.000,   # replace with your value
    'dns_start':       0.001,   # replace
    'dns_end':         0.012,   # replace
    'get_root_start':  0.013,   # replace
    'get_root_end':    0.045,   # replace
    'get_rfc_start':   2.046,   # replace
    'get_rfc_end':     2.380,   # replace
    'get_rfc2_start':  4.381,   # replace
    'get_rfc2_end':    4.710,   # replace
    'get_slow_start':  5.711,   # replace
    'get_slow_end':    8.220,   # replace
}

# Your correlation analysis here
# Compare script timestamps with first/last packet timestamps for each request


**Timestamp correlation findings:**

*(What gap did you observe between script timestamps and capture timestamps? What explains it?  
How did application-layer RTT compare to TCP-level RTT?)*

### Reflection 3.A

Your capture contains DNS traffic even though your HTTP client from Lab 1 uses
`socket.getaddrinfo` for resolution. Where does this DNS traffic come from, and what does
it tell you about how name resolution works on the JupyterHub server?

**Your answer:**

*(Write here)*

### Reflection 3.B

Look at the `/slow` endpoint sequence number plot. If you were a TCP receiver watching this stream,
at what point would you consider sending a window update, and why?
How does this relate to TCP flow control?

**Your answer:**

*(Write here)*

### Reflection 3.C

Compare the RTTs you measured at the application layer (from your Lab 1 client)
with the RTTs visible at the TCP layer in your capture.
What sources of overhead account for any difference?

**Your answer:**

*(Write here)*

---
## Submission Checklist

Before your final push, complete this checklist. Replace each `[ ]` with `[x]` when done.

- [ ] Cell 1.1: tshark commands and output shown for all 5 questions
- [ ] Cell 1.2: Wireshark approach described for all 5 questions
- [ ] Reflections 1.A and 1.B completed
- [ ] Cell 2.1: Packet size histogram plotted, statistics shown, description written
- [ ] Cell 2.2: Top 5 flows table shown
- [ ] Cell 2.3: DNS response time table shown with mean and max
- [ ] Cell 2.4: Retransmission table with delays shown
- [ ] Cell 2.5: Stream timeline table shown, overlapping streams identified
- [ ] Reflections 2.A, 2.B, and 2.C completed
- [ ] `lab2_my_capture.pcap` committed to repository
- [ ] Cell 3.1 session script ran successfully with timestamps visible
- [ ] Cell 3.2: All 4 questions answered with specific packet references
- [ ] Cell 3.3: Timestamp correlation analysis shown
- [ ] Reflections 3.A, 3.B, and 3.C completed
- [ ] Notebook runs cleanly with Kernel → Restart Kernel and Run All Cells
- [ ] At least 5 commits with descriptive messages in repository history

Final commit message must be exactly: `Lab2 final submission`

---
*CMSC395 Computer Networks - Lab 2*  
*Push this notebook to your repository with the message: `Lab2 final submission`*